In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import copy

from sklearn.linear_model import LogisticRegression,ElasticNet
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# 评价方法
from sklearn.metrics import roc_curve, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split,StratifiedKFold

import lightgbm as lgb
#from catboost import CatBoostClassifier
from tqdm import tqdm
import time 
import datetime

In [2]:
fea_df = pd.read_pickle('fea_df2.pkl')

In [3]:
data=fea_df

In [4]:
#candle_begin_time,symbol,is_spot,close
TIME='candle_begin_time'
TARGET='Return_24'
USE_COLS = data.columns.difference(['candle_begin_time','symbol','is_spot','close','Return_24'])

In [5]:
data['count'] = data.groupby('candle_begin_time')['symbol'].transform('count') - 1

In [6]:
data[TARGET] = (data.groupby('candle_begin_time')[TARGET].rank(method='min', ascending=True)-1) / data['count']

In [7]:
fea_df.groupby('candle_begin_time').size().sort_index()

candle_begin_time
2021-01-01 00:00:00     79
2021-01-01 01:00:00     79
2021-01-01 02:00:00     79
2021-01-01 03:00:00     79
2021-01-01 04:00:00     79
                      ... 
2025-02-21 18:00:00    360
2025-02-21 19:00:00    360
2025-02-21 20:00:00    360
2025-02-21 21:00:00    360
2025-02-21 22:00:00    360
Length: 36310, dtype: int64

In [8]:
# 任务
# lgb 划分训练集 验证集，在验证集上早停
# 验证集分别选择 2021 2022 2023 2024
# 统计 验证集 的 rank_ic 早停轮次  其他metric

In [9]:
for fea in USE_COLS:
    if data[fea].dtypes == 'object':
        data[fea] = data[fea].astype('float')
        print(f'{fea}已转换!')

In [10]:
1

1

In [17]:
test = data[pd.to_datetime(data[TIME]) >= pd.to_datetime('2024-01-01')].reset_index(drop=True)  # 测试集
train = data[pd.to_datetime(data[TIME]) < pd.to_datetime('2024-01-01')].reset_index(drop=True)  # 训练集

In [18]:
X_train = train[USE_COLS]
y_train = train[TARGET]
X_val = test[USE_COLS]
y_val = test[TARGET]

In [19]:
t=time.time()
lgb_train = lgb.Dataset(X_train, label = y_train)
lgb_eval = lgb.Dataset(X_val, label = y_val)
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',  # 将目标设置为回归
    'metric': 'rmse',  # 使用L2误差作为评估指标，或者你可以选择 'mae' 或 'rmse'
    'num_leaves': 128,
    'max_depth': 20,
    'min_data_in_leaf': 50,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 1,
    'lambda_l2': 1,  # 正则化，越小正则化效果越强
    'min_gain_to_split': 0.2,
    'verbose': -1,  # 关闭日志输出
    # 'device': 'gpu',  # 如果需要使用 GPU 训练，取消注释
}

# 训练回归模型
lgb_clf = lgb.train(
    params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=lgb_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 提前停止
        lgb.log_evaluation(period=50)  # 每 50 次迭代打印一次日志
    ]
)
print(time.time()-t)
preds = lgb_clf.predict(X_val)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
print('preds', 'Return_24',hourly_columns.mean())
test['random_feature'] = np.random.rand(len(test))
new_columns= test.groupby('candle_begin_time')[['random_feature', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
print('random_feature', 'Return_24',new_columns.mean())

Training until validation scores don't improve for 200 rounds
[50]	valid_0's rmse: 0.288795
[100]	valid_0's rmse: 0.288465
[150]	valid_0's rmse: 0.288379
[200]	valid_0's rmse: 0.288357
[250]	valid_0's rmse: 0.288373
[300]	valid_0's rmse: 0.288396
[350]	valid_0's rmse: 0.288425
Early stopping, best iteration is:
[188]	valid_0's rmse: 0.288356
860.8803143501282
preds Return_24 0.08828312242897218
random_feature Return_24 0.0007814813006569209


In [20]:
test = data[pd.to_datetime(data[TIME]) < pd.to_datetime('2022-01-01')].reset_index(drop=True)  # 测试集
train = data[pd.to_datetime(data[TIME]) >= pd.to_datetime('2022-01-01')].reset_index(drop=True)  # 训练集
X_train = train[USE_COLS]
y_train = train[TARGET]
X_val = test[USE_COLS]
y_val = test[TARGET]
t=time.time()
lgb_train = lgb.Dataset(X_train, label = y_train)
lgb_eval = lgb.Dataset(X_val, label = y_val)
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',  # 将目标设置为回归
    'metric': 'rmse',  # 使用L2误差作为评估指标，或者你可以选择 'mae' 或 'rmse'
    'num_leaves': 128,
    'max_depth': 20,
    'min_data_in_leaf': 50,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 1,
    'lambda_l2': 1,  # 正则化，越小正则化效果越强
    'min_gain_to_split': 0.2,
    'verbose': -1,  # 关闭日志输出
    # 'device': 'gpu',  # 如果需要使用 GPU 训练，取消注释
}

# 训练回归模型
lgb_clf = lgb.train(
    params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=lgb_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 提前停止
        lgb.log_evaluation(period=50)  # 每 50 次迭代打印一次日志
    ]
)
print(time.time()-t)
preds = lgb_clf.predict(X_val)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
print('preds', 'Return_24',hourly_columns.mean())
test['random_feature'] = np.random.rand(len(test))
new_columns= test.groupby('candle_begin_time')[['random_feature', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
print('random_feature', 'Return_24',new_columns.mean())

Training until validation scores don't improve for 200 rounds
[50]	valid_0's rmse: 0.289983
[100]	valid_0's rmse: 0.28949
[150]	valid_0's rmse: 0.28936
[200]	valid_0's rmse: 0.289314
[250]	valid_0's rmse: 0.289318
[300]	valid_0's rmse: 0.289347
[350]	valid_0's rmse: 0.28936
Early stopping, best iteration is:
[196]	valid_0's rmse: 0.289309
1159.6382312774658
preds Return_24 0.130528305338291
random_feature Return_24 -0.0008583165798809999


In [21]:
# 转换 TIME 列为 datetime 类型（如果数据中的 TIME 列还不是 datetime 类型的话）
data[TIME] = pd.to_datetime(data[TIME])
# 选择2022年整年的数据作为测试集
test = data[(data[TIME] >= pd.to_datetime('2022-01-01')) & (data[TIME] < pd.to_datetime('2023-01-01'))].reset_index(drop=True)
# 其余的数据作为训练集
train = data[~((data[TIME] >= pd.to_datetime('2022-01-01')) & (data[TIME] < pd.to_datetime('2023-01-01')))].reset_index(drop=True)

In [22]:
X_train = train[USE_COLS]
y_train = train[TARGET]
X_val = test[USE_COLS]
y_val = test[TARGET]
t=time.time()
lgb_train = lgb.Dataset(X_train, label = y_train)
lgb_eval = lgb.Dataset(X_val, label = y_val)
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',  # 将目标设置为回归
    'metric': 'rmse',  # 使用L2误差作为评估指标，或者你可以选择 'mae' 或 'rmse'
    'num_leaves': 128,
    'max_depth': 20,
    'min_data_in_leaf': 50,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 1,
    'lambda_l2': 1,  # 正则化，越小正则化效果越强
    'min_gain_to_split': 0.2,
    'verbose': -1,  # 关闭日志输出
    # 'device': 'gpu',  # 如果需要使用 GPU 训练，取消注释
}

# 训练回归模型
lgb_clf = lgb.train(
    params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=lgb_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 提前停止
        lgb.log_evaluation(period=50)  # 每 50 次迭代打印一次日志
    ]
)
print(time.time()-t)
preds = lgb_clf.predict(X_val)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
print('preds', 'Return_24',hourly_columns.mean())
test['random_feature'] = np.random.rand(len(test))
new_columns= test.groupby('candle_begin_time')[['random_feature', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
print('random_feature', 'Return_24',new_columns.mean())

Training until validation scores don't improve for 200 rounds
[50]	valid_0's rmse: 0.289624
[100]	valid_0's rmse: 0.289108
[150]	valid_0's rmse: 0.288885
[200]	valid_0's rmse: 0.28879
[250]	valid_0's rmse: 0.288747
[300]	valid_0's rmse: 0.288715
[350]	valid_0's rmse: 0.288691
[400]	valid_0's rmse: 0.288684
[450]	valid_0's rmse: 0.28868
[500]	valid_0's rmse: 0.28869
[550]	valid_0's rmse: 0.288699
[600]	valid_0's rmse: 0.288707
Early stopping, best iteration is:
[448]	valid_0's rmse: 0.288679
1645.9120483398438
preds Return_24 0.11323040934420375
random_feature Return_24 -0.00038263891015583777


In [11]:
# 转换 TIME 列为 datetime 类型（如果数据中的 TIME 列还不是 datetime 类型的话）
data[TIME] = pd.to_datetime(data[TIME])
# 选择2022年整年的数据作为测试集
test = data[(data[TIME] >= pd.to_datetime('2023-01-01')) & (data[TIME] < pd.to_datetime('2024-01-01'))].reset_index(drop=True)
# 其余的数据作为训练集
train = data[~((data[TIME] >= pd.to_datetime('2023-01-01')) & (data[TIME] < pd.to_datetime('2024-01-01')))].reset_index(drop=True)
X_train = train[USE_COLS]
y_train = train[TARGET]
X_val = test[USE_COLS]
y_val = test[TARGET]
t=time.time()
lgb_train = lgb.Dataset(X_train, label = y_train)
lgb_eval = lgb.Dataset(X_val, label = y_val)
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',  # 将目标设置为回归
    'metric': 'rmse',  # 使用L2误差作为评估指标，或者你可以选择 'mae' 或 'rmse'
    'num_leaves': 128,
    'max_depth': 20,
    'min_data_in_leaf': 50,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 1,
    'lambda_l2': 1,  # 正则化，越小正则化效果越强
    'min_gain_to_split': 0.2,
    'verbose': -1,  # 关闭日志输出
    # 'device': 'gpu',  # 如果需要使用 GPU 训练，取消注释
}

# 训练回归模型
lgb_clf = lgb.train(
    params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=lgb_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 提前停止
        lgb.log_evaluation(period=50)  # 每 50 次迭代打印一次日志
    ]
)
print(time.time()-t)
preds = lgb_clf.predict(X_val)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
print('preds', 'Return_24',hourly_columns.mean())
test['random_feature'] = np.random.rand(len(test))
new_columns= test.groupby('candle_begin_time')[['random_feature', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
print('random_feature', 'Return_24',new_columns.mean())

Training until validation scores don't improve for 200 rounds
[50]	valid_0's rmse: 0.289277
[100]	valid_0's rmse: 0.288931
[150]	valid_0's rmse: 0.288803
[200]	valid_0's rmse: 0.288757
[250]	valid_0's rmse: 0.288756
[300]	valid_0's rmse: 0.288779
[350]	valid_0's rmse: 0.288794
[400]	valid_0's rmse: 0.288805
Early stopping, best iteration is:
[214]	valid_0's rmse: 0.28875
1058.7375557422638
preds Return_24 0.08557970320342409
random_feature Return_24 0.00016741966617298386


In [12]:
1

1

In [13]:
fea_df = pd.read_pickle('fea_df2.pkl')

test = fea_df[(fea_df[TIME] >= pd.to_datetime('2023-01-01')) & (fea_df[TIME] < pd.to_datetime('2024-01-01'))].reset_index(drop=True)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
hourly_columns.mean()

np.float64(0.08557970320342409)

In [15]:
test['Return_24']

0          0.014199
1          0.002975
2          0.010584
3          0.004194
4         -0.000193
             ...   
1640272    0.054213
1640273    0.033420
1640274    0.025052
1640275    0.033468
1640276    0.006452
Name: Return_24, Length: 1640277, dtype: float32

In [16]:
y_val

0          0.648276
1          0.275862
2          0.517241
3          0.324138
4          0.206897
             ...   
1640272    0.723849
1640273    0.414226
1640274    0.288703
1640275    0.418410
1640276    0.117155
Name: Return_24, Length: 1640277, dtype: float64